#### **Imports & Config**

In [ ]:
import os
import ast
import numpy as np
import pandas as pd

MAX_FRAMES = 15
FRAME_SIZE = 43 # ball(3) + 5 off(4) + 5 def(4)
DT = 0.2 # 5Hz -> 200ms between frames
POSITION_RANK = {'PG': 0, 'SG': 1, 'SF': 2, 'PF': 3, 'C': 4}

COURT_X = 94.0
COURT_Y = 50.0
MAX_BALL_Z = 20.0
MAX_SPEED = 25.0

META_COLS = ('GAME_ID', 'EVENT_NUM', 'REBOUND_TYPE', 'DETECTION_CONFIDENCE')

# Holdout toggle
# Always run False (dev) first, then True.
# tag mirrors the tabular notebook's own suffix: dev builds the UNBALANCED sequential
# set here, and balanceMoments.py derives the _balanced set from it afterwards, reusing
# the same drawn row order as the tabular balance step.
HOLDOUT = False
MOMENT_DIR = 'moment_1'
tag = '_holdout' if HOLDOUT else '_unbalanced'

TABULAR_PATH = f'../../../data/sequences/{MOMENT_DIR}/features/tabular/sequences_fe_tabular{tag}.csv'
RAW_PATH = f'../../../data/sequences/{MOMENT_DIR}/aligned/sequences_aligned_{"holdout" if HOLDOUT else "dev"}.csv'
PLAYER_PATH = '../../../data/player/processed/player_data.csv'
OUT_DIR = f'../../../data/sequences/{MOMENT_DIR}/features/sequential/'

#### **Loading**

In [ ]:

tab = pd.read_csv(TABULAR_PATH)

assert 'GAME_ID' in tab.columns and 'EVENT_NUM' in tab.columns, (
    "GAME_ID / EVENT_NUM missing from the tabular features file"
)

print(f"Tabular rows: {len(tab)}")
tab.head()

In [ ]:
# Raw all-frames source (multiple rows per shot) - needed to rebuild the tracking tensor.
raw = pd.read_csv(RAW_PATH)
print(f"Raw frames: {len(raw)}  |  unique shots: {raw.groupby(['GAME_ID', 'EVENT_NUM']).ngroups}")

# Quick sanity on frames-per-shot
fps = raw.groupby(['GAME_ID', 'EVENT_NUM']).size()
print(f"Frames per shot - min {fps.min()}, median {int(fps.median())}, max {fps.max()}")
print(f"Shots longer than MAX_FRAMES={MAX_FRAMES}: {(fps > MAX_FRAMES).sum()} "
      f"({100 * (fps > MAX_FRAMES).mean():.1f}%) - these lose their earliest frames")

In [ ]:
# Player position + height lookup, keyed on composite (player_id, team_id).
players = pd.read_csv(PLAYER_PATH)
players.fillna(players.mean(numeric_only=True), inplace=True)

player_info = {}
for _, r in players.iterrows():
    key = (int(r['player_id']), int(r['team_id']))
    pos = str(r['position']) if pd.notna(r['position']) else ''
    rank = POSITION_RANK.get(pos, None)
    if rank is None:
        print(f"Warning: Unknown position '{pos}' for player_id={int(r['player_id'])}, team_id={int(r['team_id'])}")
    player_info[key] = {
        'position_rank': rank if rank is not None else 99,
        'height': float(r['height']) if pd.notna(r['height']) else 0.0
    }

print(f"player_info entries: {len(player_info)}")

#### **Frame Vector Builder**

In [ ]:
def build_frame_vector(frame_entries, off_team_id, prev_entries=None, dt=DT):
    ball = next((e for e in frame_entries if e[1] == -1), None)
    if ball is None:
        return None

    def sort_key(e):
        info = player_info.get((int(e[1]), int(e[0])), {'position_rank': 99, 'height': 0.0})
        return (info['position_rank'], info['height'], e[1])

    off_sorted = sorted([e for e in frame_entries if e[0] == off_team_id and e[1] != -1], key=sort_key)[:5]
    def_sorted = sorted([e for e in frame_entries if e[0] != off_team_id and e[1] != -1], key=sort_key)[:5]

    # Previous-frame positions for velocity, matched by player_id
    prev_lookup = {e[1]: e for e in prev_entries if e[1] != -1} if prev_entries else {}

    def player_vec(p):
        x, y = p[2], p[3]
        if p[1] in prev_lookup:
            prev = prev_lookup[p[1]]
            vx = (x - prev[2]) / dt
            vy = (y - prev[3]) / dt
            # Cap player SPEED at MAX_SPEED (same 25 ft/s bound as the tabular pipeline),
            # preserving direction. Guards tracking "teleports" without distorting real moves.
            speed = np.hypot(vx, vy)
            if speed > MAX_SPEED:
                vx *= MAX_SPEED / speed
                vy *= MAX_SPEED / speed
        else:
            vx, vy = 0.0, 0.0
        # Normalize onto comparable scales: positions by court dims, velocity by MAX_SPEED.
        return [x / COURT_X, y / COURT_Y, vx / MAX_SPEED, vy / MAX_SPEED]

    off_vec = [c for p in off_sorted for c in player_vec(p)]
    off_vec += [0.0] * (20 - len(off_vec))   # pad to 5 players if a team has fewer entries

    def_vec = [c for p in def_sorted for c in player_vec(p)]
    def_vec += [0.0] * (20 - len(def_vec))

    ball_vec = [ball[2] / COURT_X, ball[3] / COURT_Y,
                min(max(ball[4], 0.0), MAX_BALL_Z) / MAX_BALL_Z]
    return ball_vec + off_vec + def_vec

#### **Build Sequence Lookup**

In [ ]:
sequence_lookup = {}
no_off_team = 0
empty_seq = 0
truncated = 0

has_conf = 'DETECTION_CONFIDENCE' in raw.columns
if not has_conf:
    print("Warning: DETECTION_CONFIDENCE not in the source file - "
          "re-run sequences.py to carry it through")

for (game_id, event_num), group in raw.groupby(['GAME_ID', 'EVENT_NUM']):
    group_sorted = group.sort_values('TIME_TO_SHOT', ascending=False)
    shooter_id = group_sorted['SHOOTER_ID'].iloc[0]

    # Offensive team = shooter's team (matched in tracking, not by frame position)
    off_team_id = None
    for _, row in group_sorted.iterrows():
        for e in ast.literal_eval(row['TRACKING_DATA']):
            if e[1] == shooter_id:
                off_team_id = e[0]
                break
        if off_team_id is not None:
            break

    if off_team_id is None:
        no_off_team += 1
        continue

    parsed_frames = [ast.literal_eval(row['TRACKING_DATA']) for _, row in group_sorted.iterrows()]

    frame_vectors = []
    for i, entries in enumerate(parsed_frames):
        prev = parsed_frames[i - 1] if i > 0 else None
        vec = build_frame_vector(entries, off_team_id, prev_entries=prev)
        if vec is not None:
            frame_vectors.append(vec)

    if not frame_vectors:
        empty_seq += 1
        continue

    true_len = len(frame_vectors)
    if true_len > MAX_FRAMES:
        truncated += 1

    padding = [[0.0] * FRAME_SIZE for _ in range(max(0, MAX_FRAMES - true_len))]
    padded = (padding + frame_vectors)[-MAX_FRAMES:]   # left-pad, keep last MAX_FRAMES

    seq_len = min(true_len, MAX_FRAMES)                # real (non-pad) frames after truncation
    conf = group_sorted['DETECTION_CONFIDENCE'].iloc[0] if has_conf else None

    sequence_lookup[(int(game_id), int(event_num))] = (
        np.array(padded, dtype=np.float32), seq_len, conf
    )

print(f"Sequences built: {len(sequence_lookup)}")
print(f"Skipped (no offensive team): {no_off_team}")
print(f"Skipped (empty after build): {empty_seq}")
print(f"Truncated to MAX_FRAMES={MAX_FRAMES} (earliest frames dropped): {truncated} "
      f"({100 * truncated / max(len(sequence_lookup), 1):.1f}%)")

#### **Align to Tabular Rows**

In [ ]:
STATIC_COLS = [c for c in tab.columns if c not in META_COLS]

X_seq_list, X_static_list, y_list, meta_rows = [], [], [], []
missing = 0

for _, row in tab.iterrows():
    key = (int(row['GAME_ID']), int(row['EVENT_NUM']))
    if key not in sequence_lookup:
        missing += 1
        continue
    seq_arr, seq_len, conf = sequence_lookup[key]
    X_seq_list.append(seq_arr)
    X_static_list.append(row[STATIC_COLS].to_numpy(dtype=np.float32))
    y_list.append(int(row['REBOUND_TYPE']))
    meta_rows.append({'GAME_ID': key[0], 'EVENT_NUM': key[1],
                      'SEQ_LEN': seq_len, 'DETECTION_CONFIDENCE': conf})

X_seq = np.stack(X_seq_list).astype(np.float32)
X_static = np.stack(X_static_list).astype(np.float32)
y = np.asarray(y_list, dtype=np.int64)
meta = pd.DataFrame(meta_rows)

print(f"Tabular rows with no matching sequence (dropped): {missing}")

#### **Sanity Checks**

In [ ]:
# Shapes and alignment
print(f"X_seq:    {X_seq.shape}")
print(f"X_static: {X_static.shape}  ({len(STATIC_COLS)} features)")
print(f"y:        {y.shape}")
print(f"meta:     {meta.shape}")

assert len(X_seq) == len(X_static) == len(y) == len(meta), "Array lengths are not aligned!"
assert X_seq.shape[1:] == (MAX_FRAMES, FRAME_SIZE), "Unexpected sequence tensor shape!"

# Class balance - natural (unbalanced) distribution here; balanceMoments.py derives
# the ~50/50 _balanced sequential set from this output afterwards.
unique, counts = np.unique(y, return_counts=True)
print("Class balance:", dict(zip(unique.tolist(), counts.tolist())))

# Padding / length distribution
print("SEQ_LEN - min {}, median {}, max {}".format(
    meta['SEQ_LEN'].min(), int(meta['SEQ_LEN'].median()), meta['SEQ_LEN'].max()))
assert meta['SEQ_LEN'].min() > 0, "A sequence has zero real frames!"

raw_lens = raw.groupby(['GAME_ID', 'EVENT_NUM']).size()
print(f"Source frames per shot - min {raw_lens.min()}, max {raw_lens.max()}  "
      f"(MAX_FRAMES={MAX_FRAMES}; {truncated} sequences truncated)")

# Detection confidence carried through, for the high-confidence sensitivity analysis
if meta['DETECTION_CONFIDENCE'].notna().any():
    print("DETECTION_CONFIDENCE:", meta['DETECTION_CONFIDENCE'].value_counts().to_dict())

# NaN guard
print("NaNs - X_seq:", int(np.isnan(X_seq).sum()), "| X_static:", int(np.isnan(X_static).sum()))

In [ ]:
# Normalization check (real frames only, padding excluded)
real = np.zeros(X_seq.shape[:2], dtype=bool)
for i, n in enumerate(meta['SEQ_LEN'].to_numpy()):
    real[i, -int(n):] = True
vals = X_seq[real]

pos_idx = [0, 1] + [3 + 4 * p + k for p in range(10) for k in (0, 1)]
vel_idx = [3 + 4 * p + k for p in range(10) for k in (2, 3)]

print(f"X_seq positions - min {vals[:, pos_idx].min():.3f}, max {vals[:, pos_idx].max():.3f}")
print(f"X_seq ball_z    - min {vals[:, 2].min():.3f}, max {vals[:, 2].max():.3f}")
print(f"X_seq velocity  - min {vals[:, vel_idx].min():.3f}, max {vals[:, vel_idx].max():.3f}  (expect within [-1, 1])")
print(f"X_seq overall sd {vals.std():.3f}")
print(f"Padded fraction of tensor: {100 * (1 - real.mean()):.1f}%")

assert -0.15 <= vals[:, pos_idx].min() and vals[:, pos_idx].max() <= 1.15, \
    "Positions are far outside the court - check the normalization constants"
assert -1.0 <= vals[:, vel_idx].min() and vals[:, vel_idx].max() <= 1.0, \
    "Velocities exceed the MAX_SPEED cap - the capping in player_vec is not firing" 

#### **Saving**

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

np.save(os.path.join(OUT_DIR, f'X_seq{tag}.npy'), X_seq)
np.save(os.path.join(OUT_DIR, f'X_static{tag}.npy'), X_static)
np.save(os.path.join(OUT_DIR, f'y{tag}.npy'), y)
meta.to_csv(os.path.join(OUT_DIR, f'meta{tag}.csv'), index=False)
pd.Series(STATIC_COLS).to_csv(os.path.join(OUT_DIR, f'static_feature_names{tag}.csv'), index=False, header=False)

print(f"Saved X_seq{tag}.npy, X_static{tag}.npy, y{tag}.npy, meta{tag}.csv, static_feature_names{tag}.csv to", OUT_DIR)